# Demographics and Event Details for Olympic Fencing Medallists (1896–2024) Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Olympic fencing dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.ep4w-5p9s/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}, Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs. All entities are referenced by their `@id` fields, per FAIR² and Croissant best practice.

The FAIR² fencing dataset contains a main record set, each with named fields and columns. We'll enumerate all record sets and their fields, with `@id`s.

In [ ]:
# Croissant exposes record sets/facets via the Dataset object
# Obtain all record sets and their IDs
record_sets = list(dataset.record_sets)
print("Number of record sets:", len(record_sets))

for rs in record_sets:
    print("\nRecordSet @id:", rs['@id'])
    print("  name:", rs.get('name'))
    print("  description:", rs.get('description'))
    fields = rs.get('field', [])
    for field in fields:
        f_id = field['@id'] if isinstance(field, dict) else field
        print(f"    Field @id: {f_id}")
        # Optionally: print type
        if isinstance(field, dict):
            print(f"      name: {field.get('name')}")
            print(f"      dataType: {field.get('dataType')}")
            print(f"      description: {field.get('description')}")

## 3. Data Extraction
Extract data from a specific record set into a DataFrame for analysis. We'll reference record sets and fields strictly by their `@id`s.

**Example:** Extract the data from the primary record set, which contains medals, athlete demographics, and event details.

In [ ]:
# Choose the primary record set (typically called medals/event/athletes, by @id)
# In FAIR², record set is often: 'https://sen.science/doi/10.71728/senscience.ep4w-5p9s/fair2.json#medals'
# Let's enumerate all ids, and select by name or description
primary_recordset_id = None
for rs in record_sets:
    if rs.get('name', '').lower().startswith('medal') or rs.get('description', '').lower().find('medal') != -1:
        primary_recordset_id = rs['@id']
        break

if primary_recordset_id is None:
    # fallback, pick first
    primary_recordset_id = record_sets[0]['@id']

print("Primary record set chosen:", primary_recordset_id)

# List of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns for the primary record set
print(dataframes[primary_recordset_id].columns.tolist())
dataframes[primary_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing data, and grouping by key attributes.

Let's:
- Filter athletes older than a threshold age
- Normalize age
- Group by nation (country) or event

In [ ]:
# Inspect available numeric and categorical fields
cols = dataframes[primary_recordset_id].columns.tolist()
print("Columns:", cols)

# Identify age field by @id (could be 'age', 'http://.../age', etc.)
age_field = None
for col in cols:
    if 'age' in col.lower():
        age_field = col
        break
if age_field is None:
    # fallback: try common schema
    age_field = cols[0]
print("Using age field:", age_field)

# Filter athletes older than a threshold age
threshold = 35  # e.g., analyze older athletes
filtered_df = dataframes[primary_recordset_id][dataframes[primary_recordset_id][age_field].astype(float) > threshold]
print(f"Filtered records with {age_field} > {threshold}:")
print(filtered_df.head())

# Normalize age for filtered records
filtered_df[f"{age_field}_normalized"] = (filtered_df[age_field].astype(float) - filtered_df[age_field].mean()) / filtered_df[age_field].std()
print(f"Normalized {age_field} for filtered records:")
print(filtered_df[[age_field, f"{age_field}_normalized"]].head())

# Group by country/team/event field
group_field = None
for col in cols:
    if 'nation' in col.lower() or 'country' in col.lower() or 'noc' in col.lower():
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[age_field].mean().reset_index().rename(columns={age_field: 'mean_age'})
    print(f"Grouped data by {group_field} (mean age):")
    print(grouped_df.head())

## 5. Visualization
Visualize age distribution and nationality/event relationships for Olympic fencing medallists.

**Example:** Age histogram, mean age per nation/event.

In [ ]:
# Age distribution histogram
plt.figure(figsize=(10, 6))
dataframes[primary_recordset_id][age_field].astype(float).hist(bins=20)
plt.title('Age Distribution of Olympic Fencing Medallists')
plt.xlabel('Age')
plt.ylabel('Number of Medallists')
plt.show()

# Bar chart of mean age per nation (if available)
if group_field:
    top_n = grouped_df.sort_values(by='mean_age', ascending=False).head(10)
    plt.figure(figsize=(10, 5))
    plt.bar(top_n[group_field], top_n['mean_age'], color='skyblue')
    plt.xticks(rotation=45)
    plt.title('Top 10 Nations by Mean Age of Medallists (>35)')
    plt.xlabel('Nation')
    plt.ylabel('Mean Age')
    plt.show()

## 6. Conclusion
In this notebook, we explored demographic and event details for Olympic fencing medallists (1896–2024) using the FAIR² dataset and `mlcroissant`. We loaded the data, reviewed metadata, filtered and normalized athlete ages, and visualized age distributions and national means.

Further analyses could include temporal trends, event-based grouping, or historical comparisons. All references were made via `@id`, in compliance with FAIR² and Croissant schema best practice.